# ミニCTF（Python編） — フラグを探せ

**CTF（Capture The Flag）** は、セキュリティの世界で人気の競技です。データに隠された **「フラグ」**（`FLAG{...}`）を、プログラミングの力で見つけ出します。

## ルール

- 各チャレンジで、Pythonコードを書いてフラグを取り出します
- フラグは必ず `FLAG{...}` の形をしています
- 見つけたら `check_flag(番号, フラグ)` で答え合わせをします
- まず下の「答え合わせ用の関数」のセルを実行してください

この編は、**ループ・辞書・外部ライブラリ** を駆使する本格的な問題です。本物のCTF大会で出題されるジャンルを取り入れています。

In [ ]:
# 答え合わせ用の関数（変更しないでください）
import hashlib

_answers = {
    1: "3784b2072d8092f5327f1cce1be9c084511450a925caf753ce5a44836dc8e703",
    2: "bf90cb1467e459c69445c809d33d6abb7889263e4447c54f5eeb907781c48ee2",
    3: "5b6a624bb34f458da0168f1740a443f5ef91f64a80448f8d973ecde4eaa2a51b",
    4: "5d2727864e0548c738c4e1b15c98e3443a66c8138885ab29cbbc1f87f392df53",
    5: "d13704e840470136cb417d84732a27ac50cf3d4022c589f270d2818d723a794b",
}

def check_flag(n, flag):
    """チャレンジ番号 n と、見つけたフラグ flag を渡すと正誤を判定します。"""
    flag = str(flag).strip()
    if hashlib.sha256(flag.encode()).hexdigest() == _answers.get(n):
        print("チャレンジ{}: 正解！ おめでとう。".format(n))
    else:
        print("チャレンジ{}: 不正解。 もう一度考えてみよう。".format(n))

print("準備完了。各チャレンジに挑戦しよう。")

## チャレンジ1: 総当たり攻撃（シーザー暗号）

フラグの各文字が、文字コードで **ある一定数だけ後ろにずらされて** います（シーザー暗号）。

しかし、**ずらした数（鍵）は分かりません**。

`1` から `30` までのすべての鍵を試し（総当たり攻撃）、`FLAG{` で始まる正しい結果を見つけてください。

**ヒント**: `for` ループで鍵を変えながら復号を試します。`ord()` で文字コードに、`chr()` で文字に戻せます。`文字列.startswith("FLAG{")` で正解か判定できます。
```python
for key in range(1, 31):
    candidate = "".join(chr(ord(c) - key) for c in encoded)
    if candidate.startswith("FLAG{"):
        flag = candidate
```

In [ ]:
encoded = "MSHNiy|{lfmvyjl"

# 1〜30 の鍵を総当たりして、正しいフラグを見つけよう
flag = 

check_flag(1, flag)

## チャレンジ2: パスワード解析（辞書攻撃）

盗まれたデータベースから、フラグの **SHA-256ハッシュ値** が見つかりました。

```
bf90cb1467e459c69445c809d33d6abb7889263e4447c54f5eeb907781c48ee2
```

候補リスト `candidates` の中から、このハッシュ値に一致するものを探し出してください（辞書攻撃）。

**ヒント**: 各候補をハッシュ化して、目標のハッシュ値と比較します。
```python
import hashlib
h = hashlib.sha256(候補.encode()).hexdigest()
```
`for` ループで全候補を試し、ハッシュが一致したものがフラグです。

In [ ]:
import hashlib

target_hash = "bf90cb1467e459c69445c809d33d6abb7889263e4447c54f5eeb907781c48ee2"

candidates = [
    "FLAG{weak_pass}",
    "FLAG{not_this}",
    "FLAG{cracked}",
    "FLAG{try_again}",
    "FLAG{nope_nope}",
]

# 各候補をハッシュ化し、target_hash と一致するものを探そう
flag = 

check_flag(2, flag)

## チャレンジ3: 多重エンコード

フラグが `base64` で **複数回** エンコードされています。何回かは分かりません。

フラグ（`FLAG{` で始まる文字列）が現れるまで、繰り返しデコードしてください。

**ヒント**: `while` ループを使い、結果が `FLAG{` で始まるまでデコードを繰り返します。
```python
import base64
data = encoded
while not data.startswith("FLAG{"):
    data = base64.b64decode(data.encode()).decode()
```

In [ ]:
import base64

encoded = "VW10NFFsSXpkR3RhVjFaM1dESlNiRmt5T1d0YVdEQTk="

# FLAG{ で始まるまで繰り返しデコードしよう
flag = 

check_flag(3, flag)

## チャレンジ4: 異常検知

次の `response_times` は、サーバーの応答時間（ミリ秒）の記録です。

この中に **1つだけ、明らかに異常な値**（極端に大きい値）が紛れています。これは攻撃の痕跡です。

異常な値が **何番目（インデックス）** にあるかを突き止め、同じインデックスの `evidence` リストからフラグを取り出してください。

**ヒント**: 最大値は `max(リスト)`、その値が何番目かは `リスト.index(値)` で分かります。

In [ ]:
response_times = [120, 95, 110, 9999, 105, 130, 88]
evidence = ["timing_ok", "normal", "steady", "FLAG{anomaly}", "clean", "fine", "good"]

# 異常値の位置を特定し、evidence からフラグを取り出そう
flag = 

check_flag(4, flag)

## チャレンジ5: 隠されたメッセージ（ステガノグラフィ）

次の `tokens` は、システムが発行したアクセストークンの一覧です。

一見ただのトークンですが、**各トークンの先頭の文字** を上から順に並べると、隠されたフラグが浮かび上がります。

**ヒント**: `for` ループで各トークンの最初の文字（`token[0]`）を集め、つなげます。

In [ ]:
tokens = ["Fa12", "Lq88", "Ax03", "Gz47", "{p19", "se55", "tk21", "eow7", "gm60", "oxab", "}ze4"]

# 各トークンの先頭文字を集めてフラグを復元しよう
flag = 

check_flag(5, flag)

## 全問正解おめでとう

すべてのフラグを攻略できましたか？

総当たり攻撃、辞書攻撃によるパスワード解析、多重エンコードの解読、ログからの異常検知、ステガノグラフィ — これらはすべて、本物のCTF大会やセキュリティの実務で使われる技術です。

ここまで解けたなら、実際のCTF大会（初心者向けのものも多くあります）に挑戦する力が十分あります。